# Inspect Polymarket data

In [5]:
import polars as pl

files = ["events", "markets", "prices_history"]

for name in files:
    path = f"data/polymarket/{name}.parquet"

    # Lazy scan (no data loaded yet)
    lf = pl.scan_parquet(path)

    print("=" * 60)
    print(f"FILE: {name}.parquet")

    # Get row count efficiently
    n_rows = lf.select(pl.len()).collect().item()
    print(f"Entries: {n_rows}")

    # Get schema without reading full data
    schema = lf.collect_schema()
    print("Fields:")
    for col, dtype in schema.items():
        print(f"  {col}: {dtype}")
    print()

    if name == "events" and "category" in schema:
        counts = (
            lf
            .select("category")
            .drop_nulls()
            .group_by("category")
            .len()
            .sort("len", descending=True)
            .collect()
        )

        print("*** Event categories ***")
        for row in counts.iter_rows():
            print(f"  {row[0]}: {row[1]}")
        print()

FILE: events.parquet
Entries: 219526
Fields:
  updatedAt: String
  liquidityClob: Float64
  ticker: String
  enableNegRisk: Boolean
  closedTime: String
  tags: List(Struct({'id': String, 'label': String, 'slug': String, 'forceShow': Boolean, 'publishedAt': String, 'createdAt': String, 'updatedAt': String, 'isCarousel': Boolean, 'requiresTranslation': Boolean, 'updatedBy': Int64, 'forceHide': Boolean, 'createdBy': Int64}))
  active: Boolean
  archived: Boolean
  openInterest: Int64
  competitive: Float64
  series: List(Struct({'id': String, 'ticker': String, 'slug': String, 'title': String, 'subtitle': String, 'seriesType': String, 'recurrence': String, 'image': String, 'icon': String, 'layout': String, 'active': Boolean, 'closed': Boolean, 'archived': Boolean, 'new': Boolean, 'featured': Boolean, 'restricted': Boolean, 'publishedAt': String, 'createdBy': String, 'updatedBy': String, 'createdAt': String, 'updatedAt': String, 'commentsEnabled': Boolean, 'competitive': String, 'volume24h

In [15]:
import polars as pl
import matplotlib.pyplot as plt

path = "data/polymarket/markets.parquet"

lf = pl.scan_parquet(path)

# Ensure proper datetime type
lf = lf.with_columns(
    pl.col("startDateIso")
    .str.to_datetime(strict=False)
    .alias("start_dt")
)

parsed_check = (
    lf.select(
        pl.col("start_dt").is_not_null().sum().alias("parsed_dates")
    )
    .collect()
)

min_max = (
    lf.select(
        pl.col("start_dt").min().alias("earliest"),
        pl.col("start_dt").max().alias("latest"),
    )
    .collect()
)

earliest = min_max["earliest"][0]
latest = min_max["latest"][0]

print(f"Earliest market: {earliest}")
print(f"Latest market:   {latest}")

monthly_counts = (
    lf
    .filter(pl.col("start_dt").is_not_null())
    .with_columns(
        pl.col("start_dt").dt.truncate("6mo").alias("period")
    )
    .group_by("period")
    .len()
    .sort("period")
    .collect()
)

table = (
    lf
    .filter(pl.col("start_dt").is_not_null())
    .with_columns(
        pl.col("start_dt").dt.truncate("6mo").alias("period")
    )
    .group_by("period")
    .len()
    .sort("period")
    .collect()
)

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

print(table)

Earliest market: 2021-01-25 00:00:00
Latest market:   2026-02-25 00:00:00
shape: (11, 2)
┌─────────────────────┬────────┐
│ period              ┆ len    │
│ ---                 ┆ ---    │
│ datetime[μs]        ┆ u32    │
╞═════════════════════╪════════╡
│ 2021-01-01 00:00:00 ┆ 5      │
│ 2021-07-01 00:00:00 ┆ 1147   │
│ 2022-01-01 00:00:00 ┆ 1469   │
│ 2022-07-01 00:00:00 ┆ 4      │
│ 2023-01-01 00:00:00 ┆ 1971   │
│ 2023-07-01 00:00:00 ┆ 1130   │
│ 2024-01-01 00:00:00 ┆ 1850   │
│ 2024-07-01 00:00:00 ┆ 11182  │
│ 2025-01-01 00:00:00 ┆ 34155  │
│ 2025-07-01 00:00:00 ┆ 215663 │
│ 2026-01-01 00:00:00 ┆ 220972 │
└─────────────────────┴────────┘


In [23]:
import polars as pl
from datetime import datetime, timezone
import json

path = "data/polymarket/markets.parquet"
lf = pl.scan_parquet(path)

cutoff = pl.datetime(2026, 4, 23)

df = (
    lf
    .with_columns(
        pl.col("startDateIso").str.to_datetime(strict=False).alias("start_dt"),
        pl.col("endDateIso").str.to_datetime(strict=False).alias("end_dt"),
    )
    .filter(
        (pl.col("closed") == False) &
        (pl.col("active") == True) &
        (pl.col("end_dt") > cutoff)
    )
    .select([
        "question",
        "description",
        "start_dt",
        "end_dt"
    ])
    .drop_nulls()
    .collect()
)

sample = df.sample(n=10, seed=42)

records = []
for row in sample.iter_rows(named=True):
    records.append({
        "question": row["question"],
        "resolution_criteria": row["description"],
        "today_date": datetime.utcnow().replace(microsecond=0).isoformat(),
        "start_date": row["start_dt"].isoformat() if row["start_dt"] else None,
        "close_date": row["end_dt"].isoformat() if row["end_dt"] else None,
    })

output_path = "data/sample_markets.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"Saved {len(records)} markets to {output_path}")

Saved 10 markets to data/sample_markets.json


/var/folders/l1/swc02p9s1f1fsnsjzgsd8d8r0000gr/T/ipykernel_86196/461084406.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "today_date": datetime.utcnow().replace(microsecond=0).isoformat(),
